In [1]:
import warnings
warnings.filterwarnings("ignore")

import os
import json
import joblib
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_auc_score


In [2]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.paths import PROCESSED_DATA_DIR,REPORTS_DIR

In [3]:

from src.data_loader import get_churn_reasons, load_config

In [4]:
plt.style.use("default")

In [5]:
X_t, y_t = joblib.load(PROCESSED_DATA_DIR/"telco_clean.pkl")

In [6]:
df_t = X_t.copy()
df_t["Churn"] = y_t.values

In [7]:
print(df_t.shape)
print(f"Churn Rate : {y_t.mean():.2%}")

(7043, 22)
Churn Rate : 26.54%


In [8]:
print(df_t.isna().sum().sort_values(ascending=False).head(15))

Gender               0
Senior_Citizen       0
CLTV                 0
Churn_Score          0
Total_Charges        0
Monthly_Charges      0
Payment_Method       0
Paperless_Billing    0
Contract             0
Streaming_Movies     0
Streaming_TV         0
Tech_Support         0
Device_Protection    0
Online_Backup        0
Online_Security      0
dtype: int64


In [9]:
plt.figure(figsize=(6,4))
y_t.value_counts().plot(kind="bar",color=["steelblue","tomato"])
plt.xticks([0,1],["No Churn","Churn"],rotation=0)
plt.title("Target Distribution")
plt.tight_layout()
plt.savefig(REPORTS_DIR/"telco_target_distribution.png")
plt.close()


In [10]:
config=load_config()
reasons=get_churn_reasons(config)

2026-06-26 14:03:32.607 | INFO     | src.data_loader:get_churn_reasons:294 - Top churn reasons:
2026-06-26 14:03:32.615 | INFO     | src.data_loader:get_churn_reasons:298 - 
Churn Reason
Attitude of support person                   192
Competitor offered higher download speeds    189
Competitor offered more data                 162
Don't know                                   154
Competitor made better offer                 140
Attitude of service provider                 135
Competitor had better devices                130
Network reliability                          103
Product dissatisfaction                      102
Price too high                                98
Name: count, dtype: int64


In [11]:
print(reasons.head(15))

Churn Reason
Attitude of support person                   192
Competitor offered higher download speeds    189
Competitor offered more data                 162
Don't know                                   154
Competitor made better offer                 140
Attitude of service provider                 135
Competitor had better devices                130
Network reliability                          103
Product dissatisfaction                      102
Price too high                                98
Service dissatisfaction                       89
Lack of self-service on Website               88
Extra data charges                            57
Moved                                         53
Limited range of services                     44
Name: count, dtype: int64


In [12]:
plt.figure(figsize=(10,6))
reasons.head(15).sort_values().plot(kind="barh",color="tomato")
plt.title("Top 15 Churn Reasons")
plt.tight_layout()
plt.savefig(REPORTS_DIR/"telco_top15_reasons.png")
plt.close()

In [13]:
reason_categories={
"competitor":[
"Competitor offered higher download speeds",
"Competitor offered more data",
"Competitor made better offer",
"Competitor had better devices"],
"service_quality":[
"Attitude of support person",
"Attitude of service provider",
"Network reliability",
"Product dissatisfaction",
"Service dissatisfaction"],
"pricing":[
"Price too high",
"Extra data charges"]
}

In [14]:
summary={}
for k,v in reason_categories.items():
    count=sum(reasons.get(x,0) for x in v)
    summary[k]=count
    print(f"{k}: {count}")

plt.figure(figsize=(6,6))
plt.pie(summary.values(),labels=summary.keys(),autopct="%1.1f%%")
plt.title("Grouped Churn Reasons")
plt.savefig(REPORTS_DIR/"telco_reason_groups.png")
plt.close()

competitor: 621
service_quality: 621
pricing: 155


In [15]:
contract=df_t.groupby("Contract")["Churn"].agg(["mean","count"])
print(contract)

contract["mean"].plot(kind="bar",color="darkorange")
plt.ylabel("Churn Rate")
plt.title("Contract vs Churn")
plt.tight_layout()
plt.savefig(REPORTS_DIR/"telco_contract_churn.png")
plt.close()

                    mean  count
Contract                       
Month-to-month  0.427097   3875
One year        0.112695   1473
Two year        0.028319   1695


In [16]:
internet=df_t.groupby("Internet_Service")["Churn"].agg(["mean","count"])
print(internet)

internet["mean"].plot(kind="bar",color="steelblue")
plt.ylabel("Churn Rate")
plt.title("Internet Service vs Churn")
plt.tight_layout()
plt.savefig(REPORTS_DIR/"telco_internet_churn.png")
plt.close()

                      mean  count
Internet_Service                 
DSL               0.189591   2421
Fiber optic       0.418928   3096
No                0.074050   1526


In [17]:
print(df_t.groupby("Churn")["CLTV"].mean())

plt.figure(figsize=(8,5))
sns.boxplot(data=df_t,x="Churn",y="CLTV")
plt.title("CLTV vs Churn")
plt.tight_layout()
plt.savefig(REPORTS_DIR/"telco_cltv_boxplot.png")
plt.close()

Churn
0    4490.921337
1    4149.414660
Name: CLTV, dtype: float64


In [18]:
auc=roc_auc_score(y_t,df_t["Churn_Score"])
print(f"IBM Churn Score ROC-AUC: {auc:.4f}")

IBM Churn Score ROC-AUC: 0.9417


In [19]:
df_t["tenure_group"]=pd.cut(df_t["Tenure_Months"],
bins=[0,12,24,36,48,72],
labels=["0-12","12-24","24-36","36-48","48-72"])

tenure=df_t.groupby("tenure_group")["Churn"].agg(["mean","count"])
print(tenure)

tenure["mean"].plot(kind="bar",color="green")
plt.ylabel("Churn Rate")
plt.title("Tenure vs Churn")
plt.tight_layout()
plt.savefig(REPORTS_DIR/"telco_tenure_churn.png")
plt.close()

                  mean  count
tenure_group                 
0-12          0.476782   2175
12-24         0.287109   1024
24-36         0.216346    832
36-48         0.190289    762
48-72         0.095132   2239


In [20]:
with open(REPORTS_DIR/"churn_reason_categories.json","w") as f:
    json.dump(reason_categories,f,indent=2)

print("\nBusiness Conclusions")
print("- Month-to-month customers churn the most.")
print("- Competitor reasons dominate churn.")
print("- Service quality affects retention.")
print("- Pricing strategy is important.")
print("- High CLTV customers need priority retention.")

print("\nPhase 20 LLM Ideas")
print("• Competitor -> Price match")
print("• Service -> Escalate support")
print("• Pricing -> Discount")
print("• High CLTV -> Premium retention offer")


Business Conclusions
- Month-to-month customers churn the most.
- Competitor reasons dominate churn.
- Service quality affects retention.
- Pricing strategy is important.
- High CLTV customers need priority retention.

Phase 20 LLM Ideas
• Competitor -> Price match
• Service -> Escalate support
• Pricing -> Discount
• High CLTV -> Premium retention offer
